In [ ]:
# !pip install -U langchain
# !pip install -U langchain-cohere
# !pip install -U langchain-community
# !pip install -U langchain-text-splitters
# !pip install -U langchain-chroma
# !pip install -U pypdf
# !pip install -U chromadb
# !pip install -U gradio

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from google.colab import userdata
from google.colab import files
from langchain_cohere import ChatCohere
from langchain_cohere import CohereEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate

/tmp/ipykernel_3256/1208857428.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
key = userdata.get('coherekey')

In [ ]:
uploaded = files.upload()

pdf_paths = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]
print("Uploaded PDFs:")
for p in pdf_paths:
    print(" -", p)

Saving ISTQB_CTFL_Syllabus-v4.0.pdf to ISTQB_CTFL_Syllabus-v4.0.pdf
Uploaded PDFs:
 - ISTQB_CTFL_Syllabus-v4.0.pdf


In [ ]:
documents = []

for path in pdf_paths:
    loader = PyPDFLoader(path)
    docs = loader.load()
    for d in docs:
        d.metadata["source_file"] = path
    documents.extend(docs)
    print(f"{path}: {len(docs)} pages loaded")

print("\nTotal pages loaded across all PDFs:", len(documents))

ISTQB_CTFL_Syllabus-v4.0.pdf: 74 pages loaded

Total pages loaded across all PDFs: 74


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)
print("Total chunks:", len(chunks))

Total chunks: 302


In [ ]:
embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=key
)

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="multi_pdf_rag"
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

In [ ]:
llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    cohere_api_key=key
)

In [ ]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant answering questions about a set of uploaded PDF documents.

Answer the user's question using ONLY the context below.
If the answer cannot be found in the context, say "I don't know." Do not invent information.

Context:
{context}

Question:
{question}
""")


In [ ]:
def format_docs_with_sources(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get("source_file", "unknown file")
        page = doc.metadata.get("page", None)
        page_label = f"page {page + 1}" if isinstance(page, int) else "page unknown"
        formatted.append(f"[{source} — {page_label}]\n{doc.page_content}")
    return "\n\n".join(formatted)


def get_sources(docs):
    seen = []
    for doc in docs:
        source = doc.metadata.get("source_file", "unknown file")
        page = doc.metadata.get("page", None)
        page_label = f"p.{page + 1}" if isinstance(page, int) else "p.?"
        label = f"{source} ({page_label})"
        if label not in seen:
            seen.append(label)
    return seen


def ask(question: str, active_retriever=None):
    active_retriever = active_retriever or retriever
    docs = active_retriever.invoke(question)
    context = format_docs_with_sources(docs)
    messages = prompt.format_messages(context=context, question=question)
    response = llm.invoke(messages)
    sources = get_sources(docs)
    return response.content, sources

In [ ]:
question1 = "What is this set of documents about?"
answer, sources = ask(question1)

print("Q:", question1)
print("A:", answer)

Q: What is this set of documents about?
A: The set of documents provided is about the **ISTQB Certified Tester Foundation Level (CTFL) Syllabus, version 4.0**. It outlines the structure, content, and guidelines for the ISTQB Foundation Level certification, including details on staying current with industry changes, the level of detail in the syllabus, copyright and usage conditions, and references to standards such as ISO/IEC/IEEE 29119-3 for defect reporting. The syllabus is designed to ensure internationally consistent courses and exams for software testing professionals.


In [ ]:
question2 = "list the skills needed to pass the test"
answer, sources = ask(question2)

print("Q:", question2)
print("A:", answer)

Q: list the skills needed to pass the test
A: Based on the provided context, the essential skills required for testing, as outlined in the ISTQB CTFL Syllabus v4.0, include:

1. **Testing knowledge** (to increase effectiveness of testing, e.g., by using test techniques)  
2. **Thoroughness, carefulness, curiosity, attention to details, being methodical** (to identify defects, especially the ones that are difficult to find)  
3. **Good communication skills, active listening, being a team player** (to interact effectively with stakeholders, convey information, and report/discuss defects)  
4. **Analytical thinking, critical thinking, creativity** (to increase effectiveness of testing)  
5. **Technical knowledge** (to increase efficiency of testing, e.g., by using appropriate test tools)  
6. **Domain knowledge** (to understand and communicate with end users/business representatives)  

These skills are listed under the section **1.5.1. Generic Skills Required for Testing** on page 21 of 

In [ ]:
question3 = "list the testing positions"
answer, sources = ask(question3)

print("Q:", question3)
print("A:", answer)

Q: list the testing positions
A: Based on the provided context, the two principal roles in testing are:

1. **Test Management Role**  
2. **Testing Role**  

Additionally, the context mentions that different people may take on these roles at different times, such as:  
- Team leader  
- Test manager  
- Development manager  

It also notes that one person can take on both the testing and test management roles simultaneously.


In [ ]:
question4 = "give me a summarization for the most important info"
answer, sources = ask(question4)

print("Q:", question4)
print("A:", answer)

Q: give me a summarization for the most important info
A: Here’s a summary of the most important information from the provided context:

1. **Test Case Prioritization Strategies** (from page 49):  
   - **Risk-based prioritization**: Test cases covering the most important risks are executed first.  
   - **Coverage-based prioritization**: Test cases achieving the highest coverage (e.g., statement coverage) are executed first. A variant, **additional coverage prioritization**, focuses on the highest additional coverage for each subsequent test case.  
   - **Requirements-based prioritization**: Test cases linked to the most important requirements (as defined by stakeholders) are executed first.  

2. **Considerations for Test Execution Order** (from page 49):  
   - Test cases should ideally be ordered by priority, but dependencies between test cases or features may require lower-priority tests to be executed first.  
   - Resource availability (e.g., test tools, environments, or person

In [ ]:
question5 = "how do we learn from scratch?"
answer, sources = ask(question5)

print("Q:", question3)
print("A:", answer)

Q: list the testing positions
A: I don't know. The provided context focuses on the ISTQB Certified Tester Foundation Level syllabus, including learning objectives, cognitive levels of knowledge, and specific topics like the whole team approach and testing fundamentals. It does not provide information on how to learn from scratch in a general sense.
